## QC of DR samples (Coralina 072023 and 122021)

### Trim Galore

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-qc-%j.out  # %j = job ID

module load conda/latest

# Run qc conda env with trim galore and fastqc
conda activate qc

# Define the paths and variables
FILEPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_Coralina_combined_raw"
OUTDIR_TRIM="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/trimmed"
#mkdir -p $OUTDIR_TRIM
NSLOTS=4  

SAMPLE_NAMES_FILE="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/DR_sampleids.txt"

# Check if the file exists
if [ ! -e "$SAMPLE_NAMES_FILE" ]; then
    echo "Error: $SAMPLE_NAMES_FILE does not exist."
    exit 1
fi

# Read each line from the file and perform actions
while IFS= read -r sample_id; do
    # Form the full file names
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq.gz"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq.gz"
    
    # Ensure the input files exist before running the tools
    if [ ! -e "$input_r1" ] || [ ! -e "$input_r2" ]; then
        echo "Error: Input files do not exist for sample $sample_id"
        continue
    fi

    # Run trim_galore
    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired $input_r1 $input_r2 --fastqc -o $OUTDIR_TRIM

done < "$SAMPLE_NAMES_FILE"


# JOB-ID: 48932812 
# bash script: DR_QC.sh

this script used up all the space in work lol, need to reroute the trimmed seqs to /scratch, which I do below.

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=150G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-qc-%j.out  # %j = job ID

module load conda/latest

# Run qc conda env with trim galore and fastqc
conda activate qc

# Define the paths and variables
FILEPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_Coralina_combined_raw"
OUTDIR_TRIM="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_trimmed"
REPORTS="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/trimmed"
#mkdir -p $OUTDIR_TRIM
NSLOTS=4  

SAMPLE_NAMES_FILE="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/DR_sampleids_15-37.txt"

# Check if the file exists
if [ ! -e "$SAMPLE_NAMES_FILE" ]; then
    echo "Error: $SAMPLE_NAMES_FILE does not exist."
    exit 1
fi

# Read each line from the file and perform actions
while IFS= read -r sample_id; do
    # Form the full file names
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq.gz"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq.gz"
    
    # Ensure the input files exist before running the tools
    if [ ! -e "$input_r1" ] || [ ! -e "$input_r2" ]; then
        echo "Error: Input files do not exist for sample $sample_id"
        continue
    fi

    # Run trim_galore
    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired $input_r1 $input_r2 --fastqc -o $OUTDIR_TRIM

done < "$SAMPLE_NAMES_FILE"

mv $OUTDIR_TRIM/*trimming_report.txt $REPORTS
mv $OUTDIR_TRIM/*fastqc.html $REPORTS
mv $OUTDIR_TRIM/*fastqc.zip $REPORTS

# JOB-ID: 49028790
# bash script: DR_QC.sh

In [ ]:
cd /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/trimmed #go to trim reports
conda activate multiqc
multiqc .
conda deactivate

**need to run fastqc on the raw reads!**

### Host removal

downloaded availble DCYL, SSID, and MMEA reference genomes to /project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ and indexed them

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm-index-%j.out  # %j = job ID

module load conda/latest
conda activate assembly

#build a bowtie2 index 
bowtie2-build --threads 20 /project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Dcyl_genome/GCA_043250745.1_PSU_DCyl_1.0.0_genomic.fna /project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Dcyl_genome/Dcyl_DB

bowtie2-build --threads 20 /project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/GCA_043250775.1_PSU_SSid_1.0.0_genomic.fna /project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/Ssid_DB

conda deactivate

#Job ID: 49060466

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-mcav_hostremoval-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8

#Remove host from sample reads
#set general parameters:
SAMPLENAME="mcav"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt" 
#manually created the mcav_sampleids file from the whole sample list
RAWREADSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_trimmed"
READSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_host_removed/${SAMPLENAME}"
mkdir -p $READSPATH
EXTRAFILESPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_host_removed/${SAMPLENAME}/bams"
mkdir -p $EXTRAFILESPATH
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR"
#set step parameters 
GENOME="Mcav"
INPUTPATH="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/${GENOME}_genome"
INDEX="${GENOME}_DB"

#loop through samples
while IFS= read -r SAMPLEID; do
#re-align reads back to the index
bowtie2 -p 8 -x $INPUTPATH/$INDEX -1 $RAWREADSPATH/"${SAMPLEID}_R1_001_val_1.fq.gz" -2 $RAWREADSPATH/"${SAMPLEID}_R2_001_val_2.fq.gz" -S $EXTRAFILESPATH/"${SAMPLEID}"_mapped_and_unmapped.sam
#convert sam file from bowtie to a bam file for processing
samtools view -bS $EXTRAFILESPATH/"${SAMPLEID}"_mapped_and_unmapped.sam > $EXTRAFILESPATH/"${SAMPLEID}"_mapped_and_unmapped.bam
#extract only the reads of which both do not match against the host genome
samtools view -b -f 12 -F 256 $EXTRAFILESPATH/"${SAMPLEID}"_mapped_and_unmapped.bam > $EXTRAFILESPATH/"${SAMPLEID}"_bothReadsUnmapped.bam
# sorts the file so both mates are together and then extracts them back as .fastq.gz files
samtools sort -n -m 5G -@ 2 $EXTRAFILESPATH/"${SAMPLEID}"_bothReadsUnmapped.bam -o $EXTRAFILESPATH/"${SAMPLEID}"_bothReadsUnmapped_sorted.bam
samtools fastq -c 6 -@ 8 $EXTRAFILESPATH/"${SAMPLEID}"_bothReadsUnmapped_sorted.bam \
    -1 $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz \
    -2 $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz \
    -0 /dev/null -s /dev/null -n
 if [ $? -eq 0 ]; then
        echo "host removal completed successfully for sample: $SAMPLEID"
    else
        echo "host removal encountered an error for sample: $SAMPLEID"
        exit 1  
    fi
done < "$LISTPATH/${SAMPLELIST}"
conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 49093168
# bash script file name: DR/scripts/DR_host_removal.sh

also ran this script for other species: \
DCYL job ID: 49093179 \
SSID job ID: 49093268 \
MMEA job ID: 49186952 \
ran DLAB and PSTR together because they both use the Cnat reference. job ID: 49194522 \
Found a DLAB draft genome (uploaded to ncbi in May 2025), tried this to compare to Cnat mapping, see if worthwhile to use this genome with DLAB samples instead (it has 1112 scaffolds) job ID: 49486080


*DLAB samples mapped about 50% to Cnat genome and about 80% to DLAB draft genome so I will proceed with samples that were mapped to the DLAB draft.*

### Symbiont removal

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm-mcav-symbremoval-%j.out  # %j = job ID

module load conda/latest
conda activate assembly

#Remove symbiont and human seqs using fastq screen 
SAMPLENAME="mcav"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"  
READSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_host_removed/${SAMPLENAME}"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR"
FASTQSCREEN='/home/nikea_ulrich_uml_edu/.conda/envs/assembly/share/fastq-screen-0.16.0-0'
OUTDIR="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}"
mkdir -p "$OUTDIR"

while IFS= read -r SAMPLEID; do
$FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTDIR \
$READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
 if [ $? -eq 0 ]; then
        echo "fastq_screen completed successfully for sample: $SAMPLEID"
    else
        echo "fastq_screen encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# --nohits = output reads do not map to any genomes
done < "$LISTPATH/${SAMPLELIST}"
conda deactivate
echo "Symbiont removal: All samples processed successfully."

# JOB-ID: 49169691
# bash script file name: DR_symb_removal.sh

also ran this script for other species: \
DCYL job ID: 49171049 \
SSID job ID: 49171102 \
DLAB job ID: 51532343 \
PSTR job ID: 51533458 \
MMEA job ID: 51533461

In [ ]:
module load conda/latest
conda activate multiqc 
# in directory with fastq-screen output: /final_reads_filtered
multiqc .
conda deactivate